In [1]:
import json
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

/Users/hansinirajesh/Documents/earnings-alpha/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## step1: loading transcripts and inspecting data quality

In [2]:
with open('../data/transcripts_clean.json', 'r') as f:
    transcripts = json.load(f)

print(f"loaded {len(transcripts)} transcripts\n")
print(f"{'Ticker':<8} {'Filing ID':<35} {'Remarks (words)':<20} {'QA (words)'}")
print("-" * 80)

for t in transcripts:
    r_words = len(t['prepared_remarks'].split())
    q_words = len(t['qa_section'].split()) if t.get('qa_section') else 0
    flag = "  <-- OUTLIER (full SEC doc)" if r_words > 100_000 else ""
    print(f"{t['ticker']:<8} {t['filing_id']:<35} {r_words:<20} {q_words}{flag}")

loaded 20 transcripts

Ticker   Filing ID                           Remarks (words)      QA (words)
--------------------------------------------------------------------------------
AAPL     0000320193-25-000077                8775                 0
AAPL     0000320193-25-000071                7837                 0
AAPL     0000320193-25-000055                18471                0
AAPL     0000320193-26-000005                7605                 0
MSFT     0000950170-25-010484                9055                 0
MSFT     0001193125-25-256310                673847               0  <-- OUTLIER (full SEC doc)
MSFT     0001193125-26-027198                9246                 0
MSFT     0000950170-25-100226                8718                 0
MSFT     0000950170-25-061032                9167                 0
GOOGL    0001652044-26-000012                12469                0
GOOGL    0001652044-25-000087                10864                0
GOOGL    0001652044-25-000056              

## step2: text preprocessing 
- SEC filing picked up a lot of noise along with the press-release text, so we need to strip that before feeding the text to FinBERT so that the model can see clean financial language

In [3]:
def clean_sec_text(text, max_words=None):
    if not text or not text.strip():
        return ""

    # remove XML/HTML tags
    text = re.sub(r'<[^>]{1,200}>', ' ', text)

    # remove HTML entities
    text = re.sub(r'&[a-zA-Z0-9#]{1,10};', ' ', text)

    # remove URLs and SEC URIs
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'http://xbrl\.\S+', ' ', text)

    # remove XBRL namespace tokens (e.g. us-gaap:CommonStock, xbrli:context)
    text = re.sub(r'\b[a-zA-Z][a-zA-Z0-9_-]*:[A-Za-z]\w+\b', ' ', text)

    # remove accession-number-like strings (e.g. 0001140361-25-018400)
    text = re.sub(r'\b\d{10}-\d{2}-\d{6}\b', ' ', text)

    # collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # split into sentences and keep only those with enough alphabetic content - so like it removes numeric table rows and metadata fragments
    sentences = re.split(r'(?<=[.!?])\s+', text)
    kept = []
    for sent in sentences:
        words = sent.split()
        if len(words) < 4:
            continue
        alpha_chars = sum(c.isalpha() for c in sent)
        total_chars = max(len(sent.replace(' ', '')), 1)
        if alpha_chars / total_chars >= 0.55:
            kept.append(sent.strip())
    
    result = ' '.join(kept)

    # truncate outliers to protect against 600k word filings
    if max_words:
        result = ' '.join(result.split()[:max_words])

    return result

In [4]:
# testing the cleaner on the first APPLE transcript and also displaying metadata of before and after cleaning
raw = transcripts[0]['prepared_remarks']
cleaned = clean_sec_text(raw)
print(f"before cleaning: {len(raw.split()):,} words")
print(f"after cleaning:  {len(cleaned.split()):,} words")
print()
print("sample text - first 300 words")
print(' '.join(cleaned.split()[:300]))

before cleaning: 8,775 words
after cleaning:  6,957 words

sample text - first 300 words
.txt : 20251030 .hdr.sgml : 20251030 20251030163035 ACCESSION NUMBER: CONFORMED SUBMISSION TYPE: 8-K PUBLIC DOCUMENT COUNT: 15 CONFORMED PERIOD OF REPORT: 20251030 ITEM INFORMATION: Results of Operations and Financial Condition ITEM INFORMATION: Financial Statements and Exhibits FILED AS OF DATE: 20251030 DATE AS OF CHANGE: 20251030 FILER: COMPANY DATA: COMPANY CONFORMED NAME: Apple Inc. CENTRAL INDEX KEY: 0000320193 STANDARD INDUSTRIAL CLASSIFICATION: ELECTRONIC COMPUTERS [3571] ORGANIZATION NAME: 06 Technology EIN: 942404110 STATE OF INCORPORATION: CA FISCAL YEAR END: 0927 FILING VALUES: FORM TYPE: 8-K SEC ACT: 1934 Act SEC FILE NUMBER: 001-36743 FILM NUMBER: 251436166 BUSINESS ADDRESS: STREET 1: ONE APPLE PARK WAY CITY: CUPERTINO STATE: CA ZIP: 95014 BUSINESS PHONE: (408) 996-1010 MAIL ADDRESS: STREET 1: ONE APPLE PARK WAY CITY: CUPERTINO STATE: CA ZIP: 95014 FORMER COMPANY: FORMER CONFORMED NAM

In [5]:
# applying cleaning to all transcripts and cap outlier transcripts at 15000 words to reduce runtime
print(f"{'Ticker':<8} {'Filing ID':<35} {'Raw words':<12} {'Clean words'}")
print("-" * 70)

OUTLIER_WORD_LIMIT = 15_000

for t in transcripts:
    raw_words = len(t['prepared_remarks'].split())
    is_outlier = raw_words > 100_000
    cleaned = clean_sec_text(t['prepared_remarks'],
                             max_words=OUTLIER_WORD_LIMIT if is_outlier else None)
    t['clean_remarks'] = cleaned
    clean_words = len(cleaned.split())
    flag = "  (capped)" if is_outlier else ""
    print(f"{t['ticker']:<8} {t['filing_id']:<35} {raw_words:<12} {clean_words}{flag}")

Ticker   Filing ID                           Raw words    Clean words
----------------------------------------------------------------------
AAPL     0000320193-25-000077                8775         6957
AAPL     0000320193-25-000071                7837         6474
AAPL     0000320193-25-000055                18471        7506
AAPL     0000320193-26-000005                7605         6568
MSFT     0000950170-25-010484                9055         7202
MSFT     0001193125-25-256310                673847       8844  (capped)
MSFT     0001193125-26-027198                9246         7477
MSFT     0000950170-25-100226                8718         7231
MSFT     0000950170-25-061032                9167         7257
GOOGL    0001652044-26-000012                12469        10198
GOOGL    0001652044-25-000087                10864        8982
GOOGL    0001652044-25-000056                10393        8583
META     0001628280-26-003832                9971         7167
META     0001326801-25-000050

## step3: loading finBERT
- we are using ProsusAI/finbert which outputs three labels: positive, negative and neutral.
- we need to chunk longer texts and average them across chunks since the model's max sequence length is 512 tokens

In [6]:
MODEL_NAME = 'ProsusAI/finbert'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()  # inference mode: disable dropout

# label mapping
id2label = model.config.id2label
print(f"\nmodel labels: {id2label}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 20700.15it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



model labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


## step4: scoring function
- so, here we are creating the chunks with 480 tokens each (32 tokens space is left for CLS/SEP tokens)
- and then we run FinBERT on each chunk and get the labels after which we avreage probabilities across all chunks
- then we compute a sentiment score which ranges from -1 being the most negative to +1 being the most positive.
- then we report confidence = max(P(positive), P(neutral), P(negative)) averaged across chunks

In [7]:
def score_text_finbert(text, tokenizer, model, device,
                       chunk_size=480, stride=240, batch_size=8):

    if not text or not text.strip():
        return {
            'sentiment_score': float('nan'),
            'confidence': float('nan'),
            'prob_positive': float('nan'),
            'prob_negative': float('nan'),
            'prob_neutral': float('nan'),
            'n_chunks': 0
        }
    # tokenize the full text: we chunk manually
    tokens = tokenizer.encode(text, add_special_tokens=False) # tokenizer: HuggingFace tokenizer

    # building chunks using sliding window
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunks.append(tokens[start:end])
        if end == len(tokens):
            break
        start += stride

    # scoring chunks in batches
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(chunks), batch_size):
            batch_chunks = chunks[i:i + batch_size]

            # adding CLS/SEP tokens to pad batch to equal length
            max_len = max(len(c) for c in batch_chunks)
            input_ids_list = []
            attention_mask_list = []

            for chunk in batch_chunks:
                # CLS + chunk + SEP
                ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
                pad_len = max_len + 2 - len(ids) # +2 for CLS/SEP 
                mask = [1] * len(ids) + [0] * pad_len
                ids = ids + [tokenizer.pad_token_id] * pad_len
                input_ids_list.append(ids)
                attention_mask_list.append(mask)

            input_ids = torch.tensor(input_ids_list, dtype=torch.long).to(device)
            attention_mask = torch.tensor(attention_mask_list, dtype=torch.long).to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            all_probs.extend(probs.tolist())

    all_probs = np.array(all_probs)  # shape: (n_chunks, 3)

    # FinBERT label order: positive=0, negative=1, neutral=2
    avg_probs = all_probs.mean(axis=0)
    prob_pos, prob_neg, prob_neu = avg_probs[0], avg_probs[1], avg_probs[2]

    return {
        'sentiment_score': float(prob_pos - prob_neg),
        'confidence': float(all_probs.max(axis=1).mean()),
        'prob_positive': float(prob_pos),
        'prob_negative': float(prob_neg),
        'prob_neutral': float(prob_neu),
        'n_chunks': len(chunks)
    }


In [8]:
# sanity check
# testing on custom financial statements
test_sentences = [
    "Revenue grew 15% driven by strong cloud performance and record iPhone sales.",
    "We are deeply concerned about the macroeconomic headwinds and declining margins.",
    "Operating income was $10 billion for the quarter."
]
for sent in test_sentences:
    result = score_text_finbert(sent, tokenizer, model, device)
    print(f"Text: {sent[:60]}...") # printing first 60 characters
    print(f"  Score={result['sentiment_score']:+.3f}  "
          f"P(pos)={result['prob_positive']:.3f}  "
          f"P(neg)={result['prob_negative']:.3f}  "
          f"P(neu)={result['prob_neutral']:.3f}")
    print()

Text: Revenue grew 15% driven by strong cloud performance and reco...
  Score=+0.946  P(pos)=0.963  P(neg)=0.017  P(neu)=0.020

Text: We are deeply concerned about the macroeconomic headwinds an...
  Score=-0.957  P(pos)=0.010  P(neg)=0.966  P(neu)=0.024

Text: Operating income was $10 billion for the quarter....
  Score=+0.030  P(pos)=0.102  P(neg)=0.072  P(neu)=0.825



## step5: running FinBERT on all 20 transcripts

In [9]:
results = []

for i, t in enumerate(transcripts):
    ticker = t['ticker']
    filing_id = t['filing_id']
    clean_text = t.get('clean_remarks', '')

    print(f"[{i+1:02d}/20] {ticker} | {filing_id}")
    print(f"       text length: {len(clean_text.split()):,} words")

    # score prepared remarks (management commentary)
    remarks_score = score_text_finbert(clean_text, tokenizer, model, device)
    print(f"       remarks:  score={remarks_score['sentiment_score']:+.4f}, "
          f"chunks={remarks_score['n_chunks']}, "
          f"confidence={remarks_score['confidence']:.3f}")

    # q&a section: all are empty in this dataset
    qa_text = t.get('qa_section', '')
    clean_qa = clean_sec_text(qa_text) if qa_text else ''
    qa_score = score_text_finbert(clean_qa, tokenizer, model, device)
    if qa_score['n_chunks'] > 0:
        print(f"       q&a:      score={qa_score['sentiment_score']:+.4f}, "
              f"chunks={qa_score['n_chunks']}")
    else:
        print(f"       q&a:      empty")

    results.append({
        'ticker': ticker,
        'filing_id': filing_id,
        # remarks sentiment
        'remarks_sentiment': remarks_score['sentiment_score'],
        'remarks_confidence': remarks_score['confidence'],
        'remarks_prob_positive': remarks_score['prob_positive'],
        'remarks_prob_negative': remarks_score['prob_negative'],
        'remarks_prob_neutral': remarks_score['prob_neutral'],
        'remarks_n_chunks': remarks_score['n_chunks'],
        # q&a sentiment
        'qa_sentiment': qa_score['sentiment_score'],
        'qa_confidence': qa_score['confidence'],
        'qa_prob_positive': qa_score['prob_positive'],
        'qa_prob_negative': qa_score['prob_negative'],
        'qa_prob_neutral': qa_score['prob_neutral'],
        'qa_n_chunks': qa_score['n_chunks'],
    })
    print()

Token indices sequence length is longer than the specified maximum sequence length for this model (15190 > 512). Running this sequence through the model will result in indexing errors


[01/20] AAPL | 0000320193-25-000077
       text length: 6,957 words
       remarks:  score=-0.0435, chunks=63, confidence=0.892
       q&a:      empty

[02/20] AAPL | 0000320193-25-000071
       text length: 6,474 words
       remarks:  score=-0.0376, chunks=61, confidence=0.902
       q&a:      empty

[03/20] AAPL | 0000320193-25-000055
       text length: 7,506 words
       remarks:  score=-0.0492, chunks=117, confidence=0.887
       q&a:      empty

[04/20] AAPL | 0000320193-26-000005
       text length: 6,568 words
       remarks:  score=-0.0328, chunks=61, confidence=0.904
       q&a:      empty

[05/20] MSFT | 0000950170-25-010484
       text length: 7,202 words
       remarks:  score=-0.0360, chunks=63, confidence=0.888
       q&a:      empty

[06/20] MSFT | 0001193125-25-256310
       text length: 8,844 words
       remarks:  score=-0.0026, chunks=92, confidence=0.869
       q&a:      empty

[07/20] MSFT | 0001193125-26-027198
       text length: 7,477 words
       remarks:  sc

## step 6: inspect results

In [11]:
df = pd.DataFrame(results)
print("sentiment score for all transcripts")
display_cols = ['ticker', 'filing_id', 'remarks_sentiment', 'remarks_confidence', 'remarks_n_chunks', 'qa_n_chunks']
print(df[display_cols].to_string(index=False))
print()

# summary
print("summary stats by ticker:")
print(df.groupby('ticker')['remarks_sentiment'].agg(['mean', 'min', 'max']).round(4))

sentiment score for all transcripts
ticker            filing_id  remarks_sentiment  remarks_confidence  remarks_n_chunks  qa_n_chunks
  AAPL 0000320193-25-000077          -0.043531            0.891800                63            0
  AAPL 0000320193-25-000071          -0.037594            0.901586                61            0
  AAPL 0000320193-25-000055          -0.049160            0.887046               117            0
  AAPL 0000320193-26-000005          -0.032791            0.903634                61            0
  MSFT 0000950170-25-010484          -0.035976            0.888324                63            0
  MSFT 0001193125-25-256310          -0.002616            0.869500                92            0
  MSFT 0001193125-26-027198          -0.017730            0.872319                64            0
  MSFT 0000950170-25-100226          -0.021958            0.885304                62            0
  MSFT 0000950170-25-061032          -0.023707            0.882158                

In [12]:
# probability breakdown for each transcript
print("Probability breakdown (positive / negative / neutral):")
print("-" * 75)
print(f"{'Ticker':<8} {'Filing ID':<35} {'Positive':>10} {'Negative':>10} {'Neutral':>10}")
print("-" * 75)
for _, row in df.iterrows():
    print(f"{row['ticker']:<8} {row['filing_id']:<35} "
          f"{row['remarks_prob_positive']:>10.3f} "
          f"{row['remarks_prob_negative']:>10.3f} "
          f"{row['remarks_prob_neutral']:>10.3f}")

Probability breakdown (positive / negative / neutral):
---------------------------------------------------------------------------
Ticker   Filing ID                             Positive   Negative    Neutral
---------------------------------------------------------------------------
AAPL     0000320193-25-000077                     0.039      0.083      0.878
AAPL     0000320193-25-000071                     0.037      0.075      0.888
AAPL     0000320193-25-000055                     0.036      0.085      0.880
AAPL     0000320193-26-000005                     0.039      0.072      0.889
MSFT     0000950170-25-010484                     0.057      0.093      0.850
MSFT     0001193125-25-256310                     0.087      0.090      0.824
MSFT     0001193125-26-027198                     0.073      0.090      0.837
MSFT     0000950170-25-100226                     0.065      0.087      0.849
MSFT     0000950170-25-061032                     0.066      0.089      0.845
GOOGL    0001

In [13]:
# saving results to csv
output_path = '../data/sentiment_scores.csv'
df.to_csv(output_path, index=False)
print()
print("column dtypes:")
print(df.dtypes)
print()
print("first few rows:")
print(df[['ticker', 'filing_id', 'remarks_sentiment', 'remarks_confidence', 'remarks_n_chunks']].head(10).to_string(index=False))


column dtypes:
ticker                       str
filing_id                    str
remarks_sentiment        float64
remarks_confidence       float64
remarks_prob_positive    float64
remarks_prob_negative    float64
remarks_prob_neutral     float64
remarks_n_chunks           int64
qa_sentiment             float64
qa_confidence            float64
qa_prob_positive         float64
qa_prob_negative         float64
qa_prob_neutral          float64
qa_n_chunks                int64
dtype: object

first few rows:
ticker            filing_id  remarks_sentiment  remarks_confidence  remarks_n_chunks
  AAPL 0000320193-25-000077          -0.043531            0.891800                63
  AAPL 0000320193-25-000071          -0.037594            0.901586                61
  AAPL 0000320193-25-000055          -0.049160            0.887046               117
  AAPL 0000320193-26-000005          -0.032791            0.903634                61
  MSFT 0000950170-25-010484          -0.035976            0.888324